# Cross-modality brain panel transfer to RIBOMap

Can a compact gene panel transfer cell-type and brain-region biology from reference modalities into RIBOMap? Deep-RIBOmap and STARmap measure related but non-identical molecular views, so the biological question is whether the same panel captures stable tissue structure without erasing ribosome-associated expression differences.

This notebook passes current SMITH panels directly through held-out biological analyses. Written files are provenance outputs, not later inputs.

## Download the real input data

```bash
python scripts/download_tutorial_data.py --case 03_ribomap_transfer --data-root data/tutorials
```

## Load and prepare shared-gene brain modalities

In [ ]:
from pathlib import Path
import os, sys

ROOT = Path.cwd().resolve()
sys.path.insert(0, str(ROOT / "src"))

import anndata as ad
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from IPython import get_ipython
get_ipython().run_line_magic("matplotlib", "inline")
from reproducibility.workflows.common import ranked_genes, run_smith, write_json, write_panel_genes
from reproducibility.workflows.ribomap_transfer.evaluate_outputs import prepare_shared_adata, evaluate_panel_loaded
from reproducibility.workflows.ribomap_transfer.analysis import bias_table_from_objects, jaccard_from_panel_records, performance_paired_tests, bias_pairwise_tests
from reproducibility.workflows.ribomap_transfer.plot_figure4 import _draw_performance, _draw_jaccard, _draw_bias

DATA_ROOT = Path(os.environ.get("SMITH_TUTORIAL_DATA", "data/tutorials")).resolve()
CASE_OUTPUT = Path(os.environ.get("SMITH_TUTORIAL_OUTPUT", "outputs/tutorials")).resolve() / "ribomap"
FIGURE_DATA = CASE_OUTPUT / "figure_data"
EPOCHS = int(os.environ.get("SMITH_TUTORIAL_EPOCHS", 30))
DEVICE = os.environ.get("SMITH_TUTORIAL_DEVICE", 'cpu')
MAX_CELLS = int(os.environ.get("SMITH_TUTORIAL_MAX_CELLS", "3000"))
FIGURE_DATA.mkdir(parents=True, exist_ok=True)


In [ ]:
relative_inputs = [
    "ribomap_transfer/ribomap/deep_brain_ribomap.h5ad",
    "ribomap_transfer/ribomap/mouse_brain_starmap_rep2.h5ad",
    "ribomap_transfer/ribomap/mouse_brain_ribomap_rep2.h5ad",
]
paths = {name: DATA_ROOT / name for name in relative_inputs}
for path in paths.values():
    if not path.is_file():
        raise FileNotFoundError(path)
deep_raw, star_raw, ribomap_target = [ad.read_h5ad(paths[name]) for name in relative_inputs]
deep_shared = prepare_shared_adata(deep_raw, ribomap_target)
star_shared = prepare_shared_adata(star_raw, ribomap_target)


## Train SMITH and evaluate held-out RIBOMap biology

In [ ]:
panel_records, metric_rows, source_runs = [], [], {}
for source, source_adata in (("Deep-RIBOmap", deep_shared), ("STARmap", star_shared)):
    source_runs[source] = {}
    for training_seed in (1, 2):
        trained = run_smith(
            adata_file=None, adata=source_adata,
            output_dir=CASE_OUTPUT / "runs" / source / f"seed_{training_seed}" / "SMITH",
            tasks="recon,cls,standard_coordination" if any(key in source_adata.obsm for key in ("spatial", "X_spatial")) else "recon,cls",
            task_name=f"{source}_to_RIBOMap", panel_size=128, epochs=EPOCHS,
            device=DEVICE, seed=training_seed, batch_size=128, max_cells=MAX_CELLS,
            force=True, include_in_memory=True,
        )
        source_runs[source][training_seed] = trained
        for size in (32, 64, 128):
            genes = ranked_genes(trained["ranking_frame"], size)
            panel_path = Path(trained["output_dir"]) / "panels" / f"SMITH_top{size}.tsv"
            write_panel_genes(genes, panel_path)
            panel_records.append({
                "source": source, "method": "SMITH", "training_seed": training_seed,
                "panel_size": size, "panel_genes": genes, "panel_file": str(panel_path),
            })

for record in panel_records:
    for seed in (1, 2, 3):
        for label in ("celltype", "region"):
            result, prediction = evaluate_panel_loaded(
                ribomap_target, record["panel_genes"], record["panel_size"], seed,
                label_column=label,
                output_dir=CASE_OUTPUT / "evaluations" / record["source"] / f"panel_{record['panel_size']}" / f"seed_{seed}" / label,
            )
            metric_rows.append({
                **{key: record[key] for key in ("source", "method", "training_seed", "panel_size")},
                "evaluation_seed": seed, "label": label, **result["metrics"],
            })
metrics = pd.DataFrame(metric_rows)
metrics.to_csv(FIGURE_DATA / "figure4_c_f_values.tsv", sep="	", index=False)
performance_tests = performance_paired_tests(metrics)
performance_tests.to_csv(FIGURE_DATA / "figure4_c_f_paired_tests.tsv", sep="	", index=False)


### Figure 4c: biological transfer

In [ ]:
figure, axis = plt.subplots(figsize=(2.25, 2.25), facecolor="white")
figure_metrics = metrics[(metrics["source"] == 'Deep-RIBOmap') & (metrics["label"] == 'celltype')]
_draw_performance(axis, figure_metrics, 'Deep-RIBOmap', 'celltype', 'Deep-RIBOmap to RIBOMap', (0.0, 0.4))
figure.text(0.015, 0.985, 'c', ha="left", va="top", fontsize=10, weight="bold")
figure.subplots_adjust(left=0.26, right=0.97, bottom=0.22, top=0.86)
display(figure)
plt.close(figure)

### Figure 4d: biological transfer

In [ ]:
figure, axis = plt.subplots(figsize=(2.25, 2.25), facecolor="white")
figure_metrics = metrics[(metrics["source"] == 'Deep-RIBOmap') & (metrics["label"] == 'region')]
_draw_performance(axis, figure_metrics, 'Deep-RIBOmap', 'region', '', (0.1, 0.6))
figure.text(0.015, 0.985, 'd', ha="left", va="top", fontsize=10, weight="bold")
figure.subplots_adjust(left=0.26, right=0.97, bottom=0.22, top=0.86)
display(figure)
plt.close(figure)

### Figure 4e: biological transfer

In [ ]:
figure, axis = plt.subplots(figsize=(2.25, 2.25), facecolor="white")
figure_metrics = metrics[(metrics["source"] == 'STARmap') & (metrics["label"] == 'celltype')]
_draw_performance(axis, figure_metrics, 'STARmap', 'celltype', 'STARmap to RIBOMap', (0.1, 0.45))
figure.text(0.015, 0.985, 'e', ha="left", va="top", fontsize=10, weight="bold")
figure.subplots_adjust(left=0.26, right=0.97, bottom=0.22, top=0.86)
display(figure)
plt.close(figure)

### Figure 4f: biological transfer

In [ ]:
figure, axis = plt.subplots(figsize=(2.25, 2.25), facecolor="white")
figure_metrics = metrics[(metrics["source"] == 'STARmap') & (metrics["label"] == 'region')]
_draw_performance(axis, figure_metrics, 'STARmap', 'region', '', (0.1, 0.6))
figure.text(0.015, 0.985, 'f', ha="left", va="top", fontsize=10, weight="bold")
figure.subplots_adjust(left=0.26, right=0.97, bottom=0.22, top=0.86)
display(figure)
plt.close(figure)

### Figure 4g: same- and cross-modality panel overlap

In [ ]:
overlap = jaccard_from_panel_records(panel_records)
overlap.to_csv(FIGURE_DATA / "figure4_g_jaccard.tsv", sep="	", index=False)
figure, axis = plt.subplots(figsize=(2.15, 2.62), facecolor="white")
_draw_jaccard(axis, overlap)
figure.text(0.015, 0.985, "g", ha="left", va="top", fontsize=10, weight="bold")
figure.subplots_adjust(left=0.25, right=0.97, bottom=0.17, top=0.78)
display(figure)
plt.close(figure)

### Figure 4h: RIBOMap-specific expression bias

In [ ]:
bias = bias_table_from_objects(ribomap_target, star_raw)
bias_parts = []
for size in (32, 64, 128):
    deep = next(set(row["panel_genes"]) for row in panel_records if row["source"] == "Deep-RIBOmap" and row["training_seed"] == 1 and row["panel_size"] == size)
    star = next(set(row["panel_genes"]) for row in panel_records if row["source"] == "STARmap" and row["training_seed"] == 1 and row["panel_size"] == size)
    part = bias.copy()
    part["panel_size"], part["method"] = size, "SMITH"
    part["group"] = part["gene_symbol"].map(
        lambda gene: "RIBOMap-only" if gene in deep - star else (
            "Shared" if gene in deep & star else (
                "STARmap-only" if gene in star - deep else "Background"
            )
        )
    )
    bias_parts.append(part)
bias_values = pd.concat(bias_parts, ignore_index=True)
bias_values.to_csv(FIGURE_DATA / "figure4_h_ribomap_bias.tsv", sep="	", index=False)
bias_tests = bias_pairwise_tests(bias_values)
bias_tests.to_csv(FIGURE_DATA / "figure4_h_pairwise_tests.tsv", sep="	", index=False)
figure, axis = plt.subplots(figsize=(2.30, 2.59), facecolor="white")
_draw_bias(axis, bias_values, bias_tests)
figure.text(0.015, 0.985, "h", ha="left", va="top", fontsize=10, weight="bold")
figure.subplots_adjust(left=0.25, right=0.97, bottom=0.28, top=0.86)
display(figure)
plt.close(figure)

## Record the run

The manifest is written after analysis and is never read by this notebook.

In [ ]:
write_json(CASE_OUTPUT / "run_manifest.json", {
    "workflow": "03_ribomap_transfer", "inputs": relative_inputs,
    "configuration": {"epochs": EPOCHS, "device": DEVICE, "training_seeds": [1, 2], "panel_sizes": [32, 64, 128]},
    "outputs": {
        "metrics": str(FIGURE_DATA / "figure4_c_f_values.tsv"),
        "overlap": str(FIGURE_DATA / "figure4_g_jaccard.tsv"),
        "bias": str(FIGURE_DATA / "figure4_h_ribomap_bias.tsv"),
    },
})

## Full manuscript command

The CLI workflow remains the entry point for external baselines and repeated training seeds.

```bash
python reproducibility/workflows/ribomap_transfer/run_tutorial.py --data-root data/tutorials --output-dir outputs/paper/ribomap --methods SMITH,PERSIST-class,PERSIST,ActiveSVM,scGIST,scGeneFit,Spapros --panel-sizes 32,64,128
```